In [6]:
import pandas as pd 
import numpy as np 

ImportError: Unable to import required dependency numpy. Please see the traceback for details.

In [4]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install pandas


   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.0 MB 7.3 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/10.0 MB 5.4 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.0 MB 6.4 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/10.0 MB 6.8 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/10.0 MB 6.6 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/10.0 MB 6.8 MB/s eta 0:00:01
   ------------------------------------- -- 9.4/10.0 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 6.4 MB/s  0:00:01
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/12.6 MB 8.6 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.6 MB 8.5 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.6 MB 8.0 MB/s eta 0:00:01
   ------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
df= pd.read_csv("After_EDA_and_Feature_ENginering.csv")
df.head()

In [ ]:
df.isnull().mean().sort_values(ascending=False)

In [ ]:
df=df.drop(columns=["Fingerprint_Position","Display_Max_Brightness_nits" ,"Model_URL","Model_Image"])

In [ ]:
df['Density_g_per_cm3'] = ( df['Weight_g'] / df['Volume_cm3'])

In [ ]:
df['Flash_Type'] = (df.groupby('Brand', dropna=False)['Flash_Type'].transform(lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x))
df['Flash_Type'] = df['Flash_Type'].fillna('None')

In [ ]:
df.columns

In [ ]:
provenance_cols = [c for c in df.columns if c.endswith('_Source') or c.endswith('_is_imputed')]
df = df.drop(columns=provenance_cols)
print("Dropped:", provenance_cols)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.info()

In [ ]:
categorical_feature = [feature for feature in df.columns if df[feature].dtype in ['object', 'category', 'string']]
Numeerical_feature = [feature for feature in df.columns if df[feature].dtype in ["float64"]]

In [ ]:
import numpy as np

for col in Numeerical_feature:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        print(f"{col}: {n_inf} infinite values")
    n_huge = (df[col].abs() > 1e15).sum()
    if n_huge > 0:
        print(f"{col}: {n_huge} suspiciously huge values")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import numpy as np

df[Numeerical_feature] = df[Numeerical_feature].replace([np.inf, -np.inf], np.nan)

df = df.sort_values('Price_EUR').drop_duplicates(subset=['Brand', 'Model_Name'], keep='first')
df = df.reset_index(drop=True)

categorical_feature = [c for c in categorical_feature if c != 'Model_Name'] 

In [ ]:
feature_weights = {
    'Chipset_Generation': 2.0,
    'CPU_max_clock_ghz': 1.8,
    'GPU_Is_Flagship': 1.8,
    'RAM_GB': 1.5,
    'Storage_GB': 1.3,
    'Battery_mAh': 1.3,
    'AnTuTu_Score': 1.6,
    'GeekBench_Score': 1.4,
    'Color_Option_Count': 0.3,
    'FM_Has_RDS': 0.2,
    'FM_Can_Record': 0.2,
}

weight_vector = np.array([feature_weights.get(f, 1.0) for f in Numeerical_feature])

def apply_weights(X):
    return X * weight_vector

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('weight', FunctionTransformer(apply_weights)),
])

In [ ]:
categorical_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encode', OneHotEncoder(handle_unknown='ignore')),
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, Numeerical_feature),
    ('cat', categorical_pipeline, categorical_feature),
])

In [ ]:
full_pipeline = Pipeline([('preprocessor', preprocessor)])

In [ ]:
phone_matrix = full_pipeline.fit_transform(df)
similarity_matrix = cosine_similarity(phone_matrix)

print("Phone matrix shape:", phone_matrix.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

In [ ]:
def similar_phones(model_name, top_n=5):
    matches = df.index[df['Model_Name'] == model_name]
    if len(matches) == 0:
        return f"'{model_name}' not found in dataset"
    idx = matches[0]
    sims = similarity_matrix[idx]
    order = [i for i in np.argsort(sims)[::-1] if i != idx][:top_n]
    result = df.iloc[order][['Brand', 'Model_Name', 'Price_EUR']].copy()
    result['Similarity'] = sims[order].round(3)
    return result

In [ ]:
similar_phones("ZTE nubia RedMagic 11S Pro")

In [ ]:
similar_phones("Samsung Galaxy S25 Ultra")

In [ ]:
similar_phones("Apple iPhone 17 Pro Max")

In [ ]:
duplicate_counts = df['Model_Name'].value_counts()
print(duplicate_counts[duplicate_counts > 1])

duplicates = df[df['Model_Name'].duplicated(keep=False)].sort_values('Model_Name')

print(f"Total duplicate rows: {df['Model_Name'].duplicated(keep=False).sum()}")
print(f"Unique model names with duplicates: {(duplicate_counts > 1).sum()}")

In [ ]:
joblib.dump({
    'pipeline': full_pipeline,
    'similarity_matrix': similarity_matrix,
    'df': df,
    'feature_weights': feature_weights,
}, 'similarity_bundle.joblib')

print("Saved: similarity_bundle.joblib")